# DEPRECATED: Corporate Registry adapter training

**Do not run this notebook for Evaluator adapter training.** `CORPORATE_REGISTRY_LOOKUP` is a deterministic, comparator-only intent and does not accept a LoRA Evaluator adapter. The old registry preference corpus is retained only as legacy comparator/evaluation material.

Current Hybrid adapter work targets `B2B_IDENTITY_ENRICHMENT` / `B2B_IDENTITY_ENRICHMENT_GENERAL`. Use `notebooks/tallymark_b2b_identity_training.ipynb` instead.


In [ ]:
import os, subprocess, sys, json, pathlib, textwrap

def run(cmd, cwd=None):
    print('+', ' '.join(map(str, cmd)))
    subprocess.run(cmd, cwd=cwd, check=True)

print('Python:', sys.version)
run(['nvidia-smi'])
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda, '| available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU attached. In Kaggle settings choose Accelerator → GPU, restart the session, and rerun.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path

WORK = Path('/kaggle/working')
KIT = WORK / 'telegraph-wasm-baseline'
PROJECT = WORK / 'tallymark'

if not KIT.exists():
    run(['git', 'clone', 'https://github.com/telegraphprotocol/telegraph-wasm-baseline.git', str(KIT)])
run(['git', 'checkout', '1ed063e2f0066cfaf0e124d66ec6a17b7a6121fd'], cwd=KIT)
if not PROJECT.exists():
    run(['git', 'clone', 'https://github.com/onchaindc/tallymark.git', str(PROJECT)])

print('Kit revision:', subprocess.check_output(['git','rev-parse','HEAD'], cwd=KIT, text=True).strip())
print('TALLYMARK revision:', subprocess.check_output(['git','rev-parse','HEAD'], cwd=PROJECT, text=True).strip())


In [ ]:
# Install the kit's Python dependencies without replacing Kaggle's CUDA-enabled PyTorch.
run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'transformers>=4.48', 'peft', 'numpy', 'onnx', 'onnxruntime', 'huggingface_hub'])
import torch
assert torch.cuda.is_available(), 'CUDA disappeared after dependency installation'
print('CUDA still available:', torch.cuda.get_device_name(0))


In [ ]:
DATA = PROJECT / 'training/corporate-registry-pairs.jsonl'
assert DATA.exists(), f'Missing training data: {DATA}'

# Re-run the official pinned dataset gate before spending GPU time.
run([sys.executable, '-m', 'float_runtime.train_adapter', '--intent', 'CORPORATE_REGISTRY_LOOKUP_GENERAL', '--data', str(DATA), '--check-data'], cwd=KIT)


In [ ]:
# Train against the parent family prefix. The exported adapter will be registered on this prefix.
# The trainer's own held-out evaluation must PASS; this cell stops on FAIL.
OUT = KIT / 'runs/CORPORATE_REGISTRY_LOOKUP'
run([
    sys.executable, '-m', 'float_runtime.train_adapter',
    '--intent', 'CORPORATE_REGISTRY_LOOKUP',
    '--data', str(DATA),
    '--out', str(OUT),
    '--epochs', '3',
    '--max-length', '512',
    '--batch-size', '1',
    '--grad-accum', '16',
], cwd=KIT)

RESULT = json.loads((OUT / 'result.json').read_text())
print(json.dumps({k: RESULT.get(k) for k in ['intent','train_pairs','gate_split','n_eval','pairwise_accuracy','length_baseline','beats_length_baseline','verdict','fail_reason']}, indent=2))
if RESULT.get('verdict') != 'PASS':
    raise RuntimeError('Held-out model gate failed. Do not export or register this model.')


In [ ]:
# Download the official Foundation Model metadata and verify its published checksum file.
FOUNDATION = KIT / 'foundation'
FOUNDATION.mkdir(exist_ok=True)
release = 'https://github.com/telegraphprotocol/telegraph-foundation/releases/download/modernbert-base-lora16-v1'
for name in ['foundation.json', 'SHA256SUMS']:
    run(['curl', '-fL', '-o', str(FOUNDATION / name), f'{release}/{name}'])
run(['sha256sum', '-c', '--ignore-missing', 'SHA256SUMS'], cwd=FOUNDATION)
print((FOUNDATION / 'foundation.json').read_text())


In [ ]:
# Export the LoRA + reward head as the protocol's safetensors delta package.
run([
    sys.executable, '-m', 'float_runtime.export_adapter',
    '--run', str(OUT),
    '--intent', 'CORPORATE_REGISTRY_LOOKUP',
    '--class', 'judgment',
    '--foundation', str(FOUNDATION / 'foundation.json'),
], cwd=KIT)

PACKAGE = OUT / 'CORPORATE_REGISTRY_LOOKUP.safetensors'
assert PACKAGE.exists(), 'Export did not create the safetensors package'
print('Package:', PACKAGE, '| bytes:', PACKAGE.stat().st_size)


In [ ]:
# Telegraph's adapterbuild requires Go 1.25+. Install a local toolchain only if needed.
import re, tarfile, urllib.request
def go_version():
    try:
        out = subprocess.check_output(['go','version'], text=True, stderr=subprocess.STDOUT)
        m = re.search(r'go(\d+)\.(\d+)', out)
        return (int(m.group(1)), int(m.group(2))) if m else (0,0)
    except Exception:
        return (0,0)

if go_version() < (1,25):
    tgz = WORK / 'go1.25.4.linux-amd64.tar.gz'
    urllib.request.urlretrieve('https://go.dev/dl/go1.25.4.linux-amd64.tar.gz', tgz)
    go_root = WORK / 'go'
    if go_root.exists():
        import shutil; shutil.rmtree(go_root)
    with tarfile.open(tgz, 'r:gz') as tf:
        tf.extractall(WORK)
    os.environ['PATH'] = str(WORK / 'go/bin') + os.pathsep + os.environ['PATH']
print(subprocess.check_output(['go','version'], text=True))
assert go_version() >= (1,25), 'Go 1.25+ is required for adapterbuild'

# Run the same parser/shape/hash gate used by Telegraph nodes.
run(['go', 'run', './cmd/adapterbuild', 'lora',
     '--intent', 'CORPORATE_REGISTRY_LOOKUP',
     '--class', 'judgment',
     '--package', str(PACKAGE),
     '--foundation', str(FOUNDATION / 'foundation.json'),
     '--out', str(OUT)], cwd=KIT)


In [ ]:
# Collect the reproducible result, model package, and manifest in Kaggle's saved output folder.
ARTIFACTS = WORK / 'tallymark-artifacts'
ARTIFACTS.mkdir(exist_ok=True)
import shutil
for src in [PACKAGE, OUT / 'result.json', OUT / 'CORPORATE_REGISTRY_LOOKUP.safetensors.manifest.json']:
    if src.exists(): shutil.copy2(src, ARTIFACTS / src.name)
shutil.copy2(FOUNDATION / 'foundation.json', ARTIFACTS / 'foundation.json')
print('Artifacts ready:', [p.name for p in ARTIFACTS.iterdir()])
print('Download these from the notebook Output panel. Do not publish/register until hosting and executor verification are complete.')


## What remains after this notebook

- Host the unchanged `.safetensors` file at a stable HTTPS or IPFS URL.
- Run `adapterexec` with the exact Foundation Model and engine on a public HTTPS endpoint with at least 5 GB RAM.
- Obtain the required evaluation-pool contributor eligibility for this intent.
- Register on the parent family with the required testnet bond and fee address, then wait for the node's remote, deterministic, and hidden-competition checks.

This notebook intentionally does not expose a Kaggle session as a public server and does not make wallet transactions.